# SQLite Database Validation

This notebook validates the SQLite insurance database.

The checks include:

- table row counts;
- primary-key uniqueness;
- foreign-key integrity;
- SQL portfolio KPIs;
- comparison of SQL results with Python calculations;
- area-level insurance metrics;
- model eligibility and completeness reporting.

In [1]:
print("Kernel is working")

Kernel is working


In [2]:
import sqlite3

print("SQLite imported successfully")
print("SQLite version:", sqlite3.sqlite_version)

SQLite imported successfully
SQLite version: 3.50.4


In [3]:
from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display


pd.set_option("display.max_columns", None)
pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.6f}",
)

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

DATABASE_PATH = (
    PROJECT_ROOT
    / "database"
    / "insurance_analytics.sqlite"
)

POLICY_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "policy_analytics.csv"
)

print(f"Current directory: {CURRENT_DIRECTORY}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Database path: {DATABASE_PATH}")
print(f"Database exists: {DATABASE_PATH.exists()}")
print(f"Policy path: {POLICY_PATH}")
print(f"Policy data exists: {POLICY_PATH.exists()}")

Current directory: C:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence\notebooks
Project root: C:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence
Database path: C:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence\database\insurance_analytics.sqlite
Database exists: True
Policy path: C:\Users\Gabriel\Documents\Projects\insurance-risk-intelligence\data\processed\policy_analytics.csv
Policy data exists: True


In [4]:
connection = sqlite3.connect(DATABASE_PATH)

table_counts_query = """
SELECT
    'policies' AS table_name,
    COUNT(*) AS row_count
FROM policies

UNION ALL

SELECT
    'claims',
    COUNT(*)
FROM claims

UNION ALL

SELECT
    'orphan_claims',
    COUNT(*)
FROM orphan_claims

UNION ALL

SELECT
    'claim_count_issues',
    COUNT(*)
FROM claim_count_issues
"""

table_counts = pd.read_sql_query(
    table_counts_query,
    connection,
)

display(table_counts)

,table_name,row_count
0,policies,678013
1,claims,26444
2,orphan_claims,195
3,claim_count_issues,9123


In [5]:
foreign_key_check = pd.read_sql_query(
    "PRAGMA foreign_key_check",
    connection,
)

print(
    "Foreign-key issues:",
    len(foreign_key_check),
)

display(foreign_key_check)

Foreign-key issues: 0


,table,rowid,parent,fkid


In [6]:
sql_kpis = pd.read_sql_query(
    "SELECT * FROM v_portfolio_kpis",
    connection,
)

display(sql_kpis)

,total_policies,total_exposure_years,total_reported_claims,policies_with_claims,claim_occurrence_rate,claim_frequency,claims_per_100_exposure_years,complete_loss_policies,complete_claim_cost,average_claim_severity,pure_premium
0,678013,"358,499.445463",36102,34060,0.050235,0.100703,10.070308,668896,"59,908,673.170000","2,265.577777",169.284628


In [7]:
policy = pd.read_csv(
    POLICY_PATH,
    low_memory=False,
)


def convert_to_boolean(
    series: pd.Series,
) -> pd.Series:
    """Convert common Boolean representations."""

    if pd.api.types.is_bool_dtype(series):
        return series

    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
        .fillna(False)
        .astype(bool)
    )


for column in [
    "SeverityModelEligible",
    "PurePremiumModelEligible",
]:
    policy[column] = convert_to_boolean(
        policy[column]
    )

In [8]:
complete_loss = policy.loc[
    policy["PurePremiumModelEligible"]
]

complete_severity = policy.loc[
    policy["SeverityModelEligible"]
]

python_kpis = {
    "total_policies": len(policy),

    "total_exposure_years": (
        policy["Exposure"].sum()
    ),

    "total_reported_claims": (
        policy["ClaimNb"].sum()
    ),

    "policies_with_claims": (
        policy["HasClaim"].sum()
    ),

    "claim_occurrence_rate": (
        policy["HasClaim"].sum()
        / len(policy)
    ),

    "claim_frequency": (
        policy["ClaimNb"].sum()
        / policy["Exposure"].sum()
    ),

    "claims_per_100_exposure_years": (
        100
        * policy["ClaimNb"].sum()
        / policy["Exposure"].sum()
    ),

    "complete_loss_policies": (
        len(complete_loss)
    ),

    "complete_claim_cost": (
        complete_loss[
            "CompleteTotalClaimAmount"
        ].sum()
    ),

    "average_claim_severity": (
        complete_severity[
            "CompleteTotalClaimAmount"
        ].sum()
        / complete_severity[
            "ClaimNb"
        ].sum()
    ),

    "pure_premium": (
        complete_loss[
            "CompleteTotalClaimAmount"
        ].sum()
        / complete_loss[
            "Exposure"
        ].sum()
    ),
}

python_kpis

{'total_policies': 678013,
 'total_exposure_years': np.float64(358499.4454629838),
 'total_reported_claims': np.int64(36102),
 'policies_with_claims': np.int64(34060),
 'claim_occurrence_rate': np.float64(0.05023502499214617),
 'claim_frequency': np.float64(0.10070308464041305),
 'claims_per_100_exposure_years': np.float64(10.070308464041306),
 'complete_loss_policies': 668896,
 'complete_claim_cost': np.float64(59908673.169999994),
 'average_claim_severity': np.float64(2265.5777774836442),
 'pure_premium': np.float64(169.2846276272367)}

In [9]:
sql_kpi_row = sql_kpis.iloc[0].to_dict()

comparison_rows = []

for metric, python_value in python_kpis.items():
    sql_value = sql_kpi_row[metric]

    passed = np.isclose(
        python_value,
        sql_value,
        rtol=1e-9,
        atol=1e-6,
    )

    comparison_rows.append(
        {
            "Metric": metric,
            "PythonValue": python_value,
            "SQLValue": sql_value,
            "AbsoluteDifference": abs(
                python_value - sql_value
            ),
            "Passed": passed,
        }
    )

kpi_comparison = pd.DataFrame(
    comparison_rows
)

display(kpi_comparison)

print(
    "All KPI comparisons passed:",
    kpi_comparison["Passed"].all(),
)

,Metric,PythonValue,SQLValue,AbsoluteDifference,Passed
0,total_policies,"678,013.000000","678,013.000000",0.000000,True
1,total_exposure_years,"358,499.445463","358,499.445463",0.000000,True
2,total_reported_claims,"36,102.000000","36,102.000000",0.000000,True
3,policies_with_claims,"34,060.000000","34,060.000000",0.000000,True
4,claim_occurrence_rate,0.050235,0.050235,0.000000,True
5,claim_frequency,0.100703,0.100703,0.000000,True
6,claims_per_100_exposure_years,10.070308,10.070308,0.000000,True
7,complete_loss_policies,"668,896.000000","668,896.000000",0.000000,True
8,complete_claim_cost,"59,908,673.170000","59,908,673.170000",0.000000,True
9,average_claim_severity,"2,265.577777","2,265.577777",0.000000,True


All KPI comparisons passed: True


In [10]:
area_metrics = pd.read_sql_query(
    """
    SELECT *
    FROM v_area_metrics
    ORDER BY claims_per_100_exposure_years DESC
    """,
    connection,
)

display(area_metrics)

,Area,policies,exposure,claims,policies_with_claim,claim_occurrence_rate,claim_frequency,claims_per_100_exposure_years,average_claim_severity,pure_premium,complete_loss_percentage
0,F,17954,"8,129.234074",1131,1055,0.058761,0.139127,13.912750,"1,524.039302",147.160554,98.095132
1,E,137167,"63,819.314270",7805,7296,0.053191,0.122298,12.229840,"2,126.335560",205.894081,98.834268
2,D,151596,"77,120.191692",8428,7892,0.052059,0.109284,10.928396,"2,243.450183",189.863933,98.779651
3,C,191880,"104,449.003785",9875,9356,0.048760,0.094544,9.454375,"2,060.069425",141.851663,98.631436
4,B,75459,"43,012.323931",3800,3630,0.048106,0.088347,8.834677,"3,370.292260",209.409120,98.534303
5,A,103957,"61,969.377712",5063,4831,0.046471,0.081702,8.170164,"2,300.722554",126.939198,98.466674


In [11]:
model_eligibility = pd.read_sql_query(
    "SELECT * FROM v_model_eligibility",
    connection,
)

claim_completeness = pd.read_sql_query(
    """
    SELECT *
    FROM v_claim_completeness
    ORDER BY policy_count DESC
    """,
    connection,
)

print("Model eligibility:")
display(model_eligibility)

print("Claim completeness:")
display(claim_completeness)

Model eligibility:


,model_sample,eligible_policies,eligible_percentage
0,Frequency model,678013,100.000000
1,Severity model,24943,3.678838
2,Pure premium model,668896,98.655336


Claim completeness:


,completeness_status,policy_count,percentage
0,no_claim_no_severity,643953,94.976498
1,matched_positive_claims,24943,3.678838
2,claim_without_severity,9116,1.344517
3,fewer_severity_rows_than_claimnb,1,0.000147


In [12]:
orphan_summary = pd.read_sql_query(
    """
    SELECT
        COUNT(*) AS orphan_claim_rows,
        COUNT(DISTINCT IDpol)
            AS orphan_policy_ids,
        SUM(ClaimAmount)
            AS orphan_claim_amount
    FROM orphan_claims
    """,
    connection,
)

display(orphan_summary)

,orphan_claim_rows,orphan_policy_ids,orphan_claim_amount
0,195,6,"788,714.180000"


In [13]:
connection.close()

print("Database connection closed.")

Database connection closed.
